# Parameter-Efficient Fine-Tuning: LoRA, Prefix and Prompt Tuning

The full code of [Parameter-Efficient Fine-Tuning: LoRA, Prefix and Prompt Tuning](https://residual-stream.lvtan71.workers.dev/posts/parameter-efficient-fine-tuning-lora/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/parameter-efficient-fine-tuning-lora) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/parameter-efficient-fine-tuning-lora.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 42 min end to end on the RTX 3090 machine that saved the outputs below.
Trains eleven LoRA adapters and reruns two of the full fine-tunes from Fine-Tuning, and What It Destroys on Qwen2.5-0.5B, 400 steps each. The table is made by a short script the post doesn't show, included here.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 42 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.18.*" huggingface_hub pandas pyarrow

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "parameter-efficient-fine-tuning-lora"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/parameter-efficient-fine-tuning-lora


## `memory.py`

LoRA's trainable-parameter count and training memory for Llama 3.1 8B: full fine-tune, LoRA and QLoRA at rank 16, and the size of one adapter. Plain arithmetic.

In [2]:
!{sys.executable} memory.py

Llama 3.1 8B: 8.03B parameters, 6.98B of them in the 7 matrices of each block
  LoRA r =  1 on all 7:    2.6M trainable, 0.03% of the model
  LoRA r =  8 on all 7:   21.0M trainable, 0.26% of the model
  LoRA r = 16 on all 7:   41.9M trainable, 0.52% of the model
  LoRA r = 64 on all 7:  167.8M trainable, 2.09% of the model

Memory for training, weights + gradients + AdamW's two moments (activations not counted):
  full fine-tune, mixed precision (16 bytes per parameter, Initialization and Optimizers):  128.5 GB
  LoRA r = 16: 16-bit frozen weights 16.1 GB + 16 bytes per LoRA parameter 0.67 GB =   16.7 GB
  QLoRA r = 16: 4-bit block weights, the rest 16-bit 5.6 GB + LoRA 0.67 GB =    6.3 GB
  (plus NF4's block scales: 0.127 bits per weight with double quantization, 0.11 GB)

One r = 16 adapter stored in 16 bits: 84 MB, against 16.1 GB for a full copy


The post printed:

```text
Llama 3.1 8B: 8.03B parameters, 6.98B of them in the 7 matrices of each block
  LoRA r =  1 on all 7:    2.6M trainable, 0.03% of the model
  LoRA r =  8 on all 7:   21.0M trainable, 0.26% of the model
  LoRA r = 16 on all 7:   41.9M trainable, 0.52% of the model
  LoRA r = 64 on all 7:  167.8M trainable, 2.09% of the model

Memory for training, weights + gradients + AdamW's two moments (activations not counted):
  full fine-tune, mixed precision (16 bytes per parameter, Initialization and Optimizers):  128.5 GB
  LoRA r = 16: 16-bit frozen weights 16.1 GB + 16 bytes per LoRA parameter 0.67 GB =   16.7 GB
  QLoRA r = 16: 4-bit block weights, the rest 16-bit 5.6 GB + LoRA 0.67 GB =    6.3 GB
  (plus NF4's block scales: 0.127 bits per weight with double quantization, 0.11 GB)

One r = 16 adapter stored in 16 bits: 84 MB, against 16.1 GB for a full copy
```

## `fetch_wiki.py`

Downloads one shard each of English and Vietnamese Wikipedia (the November 2023 dump) into `wiki/`.

In [3]:
!{sys.executable} fetch_wiki.py

## `wiki_data.py`

Builds `wiki_replay.txt` and `vi_test.txt` from `wiki/`, which `ft.py` reads.

In [4]:
!{sys.executable} wiki_data.py

WikiText test articles: 7, of which 0 are in this shard and dropped; replay text 4,001,877 chars from 1101 articles


The post printed:

```text
WikiText test articles: 7, of which 0 are in this shard and dropped; replay text 4,001,877 chars from 1101 articles
```

## `full_ft.py`

Reruns the full fine-tune of "Fine-Tuning, and What It Destroys" at lr 1e-5 without replay, seeds 0 and 1, with `ft.py`; seed 0 saves its block weights to `ft_weights.pt`. Keeps each run's summary line in `full_ft.log`. Its printed lines were not kept, so it has no `.out`.

In [5]:
!{sys.executable} full_ft.py

lr 1e-05, replay 0%, seed 0 (160 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.113, wiki 0.990, vi 1.028 | step 50: blog 1.086, wiki 1.002, vi 1.035 | step 100: blog 1.072, wiki 1.013, vi 1.042 | step 200: blog 1.070, wiki 1.031, vi 1.055 | step 400: blog 1.068, wiki 1.048, vi 1.072


lr 1e-05, replay 0%, seed 1 (154 s): step 0: blog 1.163, wiki 0.981, vi 1.022 | step 25: blog 1.113, wiki 0.990, vi 1.030 | step 50: blog 1.092, wiki 1.004, vi 1.035 | step 100: blog 1.073, wiki 1.012, vi 1.042 | step 200: blog 1.066, wiki 1.027, vi 1.052 | step 400: blog 1.086, wiki 1.052, vi 1.073


## `lora_runs.py`

Runs `lora.py` eleven times: ranks 1, 8 and 64 at 1e-3 and rank 8 at 2e-4, seeds 0 and 1 (`lora.log`), then rank 64 at 1.25e-4 twice and rank 1 at 2e-4 once (`lora_extra.log`). No `.out`, as above.

In [6]:
!{sys.executable} lora_runs.py

rank  1, lr 0.001, seed 0: 0.55M trainable of 495M (0.11%), 122 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.118, wiki 1.000 | step 50: blog 1.097, wiki 1.018 | step 100: blog 1.087, wiki 1.029 | step 200: blog 1.089, wiki 1.056 | step 400: blog 1.088, wiki 1.075 | merged vs unmerged logits, max difference 1.5e-04


rank  8, lr 0.001, seed 0: 4.40M trainable of 498M (0.88%), 127 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.113, wiki 1.032 | step 50: blog 1.099, wiki 1.053 | step 100: blog 1.100, wiki 1.087 | step 200: blog 1.121, wiki 1.142 | step 400: blog 1.153, wiki 1.205 | merged vs unmerged logits, max difference 3.2e-04


rank 64, lr 0.001, seed 0: 35.19M trainable of 529M (6.65%), 131 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.244, wiki 1.206 | step 50: blog 1.321, wiki 1.407 | step 100: blog 1.727, wiki 1.672 | step 200: blog 1.437, wiki 1.787 | step 400: blog 1.538, wiki 2.038 | merged vs unmerged logits, max difference 1.5e-04


rank  8, lr 0.0002, seed 0: 4.40M trainable of 498M (0.88%), 125 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.120, wiki 0.998 | step 50: blog 1.096, wiki 1.012 | step 100: blog 1.083, wiki 1.025 | step 200: blog 1.081, wiki 1.049 | step 400: blog 1.075, wiki 1.066 | merged vs unmerged logits, max difference 9.8e-05


rank  1, lr 0.001, seed 1: 0.55M trainable of 495M (0.11%), 127 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.117, wiki 1.000 | step 50: blog 1.100, wiki 1.022 | step 100: blog 1.089, wiki 1.034 | step 200: blog 1.087, wiki 1.051 | step 400: blog 1.100, wiki 1.085 | merged vs unmerged logits, max difference 1.0e-04


...

rank 64, lr 0.001, seed 1: 35.19M trainable of 529M (6.65%), 125 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.269, wiki 1.191 | step 50: blog 1.342, wiki 1.354 | step 100: blog 1.419, wiki 1.535 | step 200: blog 1.445, wiki 1.726 | step 400: blog 1.553, wiki 2.023 | merged vs unmerged logits, max difference 1.2e-04


rank  8, lr 0.0002, seed 1: 4.40M trainable of 498M (0.88%), 128 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.117, wiki 0.996 | step 50: blog 1.101, wiki 1.017 | step 100: blog 1.083, wiki 1.027 | step 200: blog 1.074, wiki 1.045 | step 400: blog 1.085, wiki 1.074 | merged vs unmerged logits, max difference 1.7e-04


rank 64, lr 0.000125, seed 0: 35.19M trainable of 529M (6.65%), 126 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.117, wiki 1.043 | step 50: blog 1.092, wiki 1.055 | step 100: blog 1.083, wiki 1.063 | step 200: blog 1.087, wiki 1.103 | step 400: blog 1.096, wiki 1.135 | merged vs unmerged logits, max difference 1.6e-04


rank 64, lr 0.000125, seed 1: 35.19M trainable of 529M (6.65%), 128 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.118, wiki 1.043 | step 50: blog 1.094, wiki 1.050 | step 100: blog 1.083, wiki 1.060 | step 200: blog 1.081, wiki 1.092 | step 400: blog 1.125, wiki 1.148 | merged vs unmerged logits, max difference 2.6e-04


rank  1, lr 0.0002, seed 0: 0.55M trainable of 495M (0.11%), 124 s | step 0: blog 1.163, wiki 0.981 | step 25: blog 1.140, wiki 0.981 | step 50: blog 1.118, wiki 0.989 | step 100: blog 1.102, wiki 1.000 | step 200: blog 1.090, wiki 1.020 | step 400: blog 1.073, wiki 1.037 | merged vs unmerged logits, max difference 2.8e-04


## `lora_table.py`

Turns `lora.log`, `lora_extra.log` and `full_ft.log` into the post's table of bits per byte at each checkpoint, the run times, and the merged-vs-unmerged logit check.

In [7]:
!{sys.executable} lora_table.py

before training: blog 1.163, WikiText 0.981 bits per byte

bits per byte after each step, seed 0 then seed 1 (rank 1 at 2e-4: one seed)
rank  lr            step 25       step 50       step 100      step 200      step 400
   1  0.001     blog  1.118 1.117   1.097 1.100   1.087 1.089   1.089 1.087   1.088 1.100
   1  0.001     wiki  1.000 1.000   1.018 1.022   1.029 1.034   1.056 1.051   1.075 1.085
   8  0.001     blog  1.113 1.111   1.099 1.100   1.100 1.101   1.121 1.115   1.153 1.185
   8  0.001     wiki  1.032 1.033   1.053 1.063   1.087 1.075   1.142 1.145   1.205 1.236
  64  0.001     blog  1.244 1.269   1.321 1.342   1.727 1.419   1.437 1.445   1.538 1.553
  64  0.001     wiki  1.206 1.191   1.407 1.354   1.672 1.535   1.787 1.726   2.038 2.023
   8  0.0002    blog  1.120 1.117   1.096 1.101   1.083 1.083   1.081 1.074   1.075 1.085
   8  0.0002    wiki  0.998 0.996   1.012 1.017   1.025 1.027   1.049 1.045   1.066 1.074
  64  0.000125  blog  1.117 1.118   1.092 1.094   1.083 1.0

The post printed:

```text
before training: blog 1.163, WikiText 0.981 bits per byte

bits per byte after each step, seed 0 then seed 1 (rank 1 at 2e-4: one seed)
rank  lr            step 25       step 50       step 100      step 200      step 400
   1  0.001     blog  1.118 1.117   1.097 1.101   1.088 1.089   1.090 1.087   1.088 1.100
   1  0.001     wiki  1.000 1.000   1.018 1.022   1.029 1.034   1.056 1.051   1.074 1.084
   8  0.001     blog  1.113 1.111   1.100 1.100   1.099 1.101   1.118 1.116   1.150 1.179
   8  0.001     wiki  1.032 1.033   1.053 1.062   1.087 1.075   1.142 1.149   1.211 1.229
  64  0.001     blog  1.244 1.260   1.405 1.306   1.446 1.415   1.474 1.425   1.492 1.495
  64  0.001     wiki  1.214 1.198   1.437 1.298   1.594 1.494   1.767 1.687   1.965 2.001
   8  0.0002    blog  1.119 1.117   1.096 1.100   1.083 1.083   1.082 1.074   1.075 1.084
   8  0.0002    wiki  0.998 0.997   1.012 1.016   1.025 1.027   1.050 1.045   1.066 1.074
  64  0.000125  blog  1.117 1.118   1.092 1.094   1.083 1.083   1.087 1.082   1.095 1.123
  64  0.000125  wiki  1.043 1.043   1.055 1.049   1.063 1.061   1.102 1.092   1.135 1.147
   1  0.0002    blog  1.140         1.118         1.102         1.090         1.073
   1  0.0002    wiki  0.981         0.989         1.000         1.020         1.036
full, 1e-5      blog  1.112 1.113   1.086 1.092   1.072 1.072   1.069 1.065   1.068 1.084
full, 1e-5      wiki  0.990 0.990   1.002 1.004   1.012 1.011   1.031 1.026   1.047 1.050

seconds per 400-step run: 1481 to 2100
merged vs unmerged, largest logit difference over all 11 LoRA runs: 6.2e-04
```

## `spectrum.py ft_weights.pt "lr 1e-5, no replay, 400 steps, seed 0"`

For each block matrix of the full fine-tune, the share of the update's squared size captured by its best rank-1, 8 and 64 approximations, the rank needed for 90%, and the same for a random matrix. Arguments: the weights file and a label.

In [8]:
!{sys.executable} spectrum.py ft_weights.pt "lr 1e-5, no replay, 400 steps, seed 0"

update from the full fine-tune of Fine-Tuning, and What It Destroys (lr 1e-5, no replay, 400 steps, seed 0), averaged over the 24 blocks;
share of the update's squared size captured by the best rank-k matrix, and the rank needed for 90%

matrix          shape   rank 1     rank 8     rank 64    rank for 90%   random matrix, rank 8 / 64


q_proj      896x896         6.6%      25.6%      63.6%      228 of 896     3.4% / 23.6%


k_proj      128x896        13.6%      45.9%      91.2%       60 of 128    11.0% / 65.9%


v_proj      128x896         7.0%      26.5%      80.3%       86 of 128    10.9% / 65.7%


o_proj      896x896         5.5%      20.1%      60.3%      239 of 896     3.4% / 23.6%


gate_proj  4864x896         5.5%      12.9%      36.0%      530 of 896     1.8% / 13.2%


up_proj    4864x896         4.8%      12.8%      38.1%      496 of 896     1.8% / 13.2%


down_proj   896x4864        2.4%       9.3%      33.3%      540 of 896     1.8% / 13.2%


The post printed:

```text
update from the full fine-tune of Fine-Tuning, and What It Destroys (lr 1e-5, no replay, 400 steps, seed 0), averaged over the 24 blocks;
share of the update's squared size captured by the best rank-k matrix, and the rank needed for 90%

matrix          shape   rank 1     rank 8     rank 64    rank for 90%   random matrix, rank 8 / 64
q_proj      896x896         6.6%      25.6%      63.6%      228 of 896     3.4% / 23.6%
k_proj      128x896        13.6%      45.9%      91.2%       60 of 128    11.0% / 65.9%
v_proj      128x896         6.9%      26.3%      80.3%       86 of 128    10.9% / 65.7%
o_proj      896x896         5.5%      20.0%      60.2%      240 of 896     3.4% / 23.6%
gate_proj  4864x896         5.4%      12.8%      35.9%      530 of 896     1.8% / 13.2%
up_proj    4864x896         4.8%      12.7%      38.0%      496 of 896     1.8% / 13.2%
down_proj   896x4864        2.4%       9.2%      33.2%      540 of 896     1.8% / 13.2%
```

## `truncated_update.py`

Rebuilds the fine-tuned model with each matrix's update cut to rank 0, 1, 8 and 64, and measures held-out blog and WikiText bits per byte.

In [9]:
!{sys.executable} truncated_update.py

full fine-tune's update cut to rank k in every block matrix (bits per byte)


  pretrained: blog 1.163, wiki 0.981


           0: blog 1.162, wiki 0.981


           1: blog 1.122, wiki 0.998


           8: blog 1.104, wiki 1.011


          64: blog 1.079, wiki 1.026


         all: blog 1.068, wiki 1.048


The post printed:

```text
full fine-tune's update cut to rank k in every block matrix (bits per byte)
  pretrained: blog 1.163, wiki 0.981
           0: blog 1.162, wiki 0.981
           1: blog 1.122, wiki 0.998
           8: blog 1.103, wiki 1.010
          64: blog 1.078, wiki 1.026
         all: blog 1.068, wiki 1.047
```